# Notebook 01 — DICOM → PNG Preprocessing

For each patient:
1. Load CT series → Hounsfield Units
2. Apply lung windowing → uint8
3. Extract GTV-1 tumor mask (RTSTRUCT, fallback to SEG)
4. Crop around tumor, resize to 224×224, save as PNG
5. Build `slices_index.csv`

In [ ]:
import os, sys, logging, time
import pandas as pd
import numpy as np

# Setup paths
sys.path.insert(0, os.path.abspath('.'))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger('nb01')

In [ ]:
# ── Configuration ────────────────────────────────────────────────
DATASET_ROOT = '/path/to/local/resource'
OUTPUT_DIR   = 'processed_data'

# Windowing
WINDOW_CENTER = -600.0   # Lung window
WINDOW_WIDTH  = 1500.0

# Crop settings
CROP_MARGIN  = 30        # pixels around tumor bbox
TARGET_SIZE  = (224, 224)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Discover patients ────────────────────────────────────────────
patients = sorted([d for d in os.listdir(DATASET_ROOT)
                    if d.startswith('LUNG1-') and
                    os.path.isdir(os.path.join(DATASET_ROOT, d))])
print(f'Found {len(patients)} patients')

In [ ]:
# ── Process all patients ─────────────────────────────────────────
from utils.preprocess_utils import process_patient

all_records = []
failed_patients = []
start = time.time()

for idx, patient_id in enumerate(patients):
    patient_dir = os.path.join(DATASET_ROOT, patient_id)
    try:
        records = process_patient(
            patient_id=patient_id,
            patient_dir=patient_dir,
            output_dir=OUTPUT_DIR,
            window_center=WINDOW_CENTER,
            window_width=WINDOW_WIDTH,
            crop_margin=CROP_MARGIN,
            target_size=TARGET_SIZE,
        )
        all_records.extend(records)
    except Exception as e:
        logger.error(f'{patient_id} FAILED: {e}')
        failed_patients.append((patient_id, str(e)))

    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start
        logger.info(f'Progress: {idx+1}/{len(patients)} '
                     f'({elapsed:.0f}s, {len(all_records)} slices)')

elapsed = time.time() - start
print(f'\n✅ Done: {len(patients) - len(failed_patients)}/{len(patients)} '
      f'patients processed in {elapsed:.0f}s')
print(f'   Total slices saved: {len(all_records)}')
print(f'   Tumor slices: {sum(r["has_tumor"] for r in all_records)}')
if failed_patients:
    print(f'\n⚠️  Failed patients ({len(failed_patients)}):')
    for pid, err in failed_patients:
        print(f'   {pid}: {err}')

In [ ]:
# ── Save slices_index.csv ────────────────────────────────────────
df = pd.DataFrame(all_records)
csv_path = os.path.join(OUTPUT_DIR, 'slices_index.csv')
df.to_csv(csv_path, index=False)
print(f'Saved {len(df)} rows to {csv_path}')
df.head(10)

In [ ]:
# ── Summary statistics ───────────────────────────────────────────
print('Slices per patient:')
print(df.groupby('patient_id').size().describe())
print(f'\nTumor slice ratio: {df["has_tumor"].mean():.2%}')

if failed_patients:
    fail_df = pd.DataFrame(failed_patients, columns=['patient_id', 'error'])
    fail_df.to_csv(os.path.join(OUTPUT_DIR, 'failed_patients.csv'), index=False)
    print(f'\nFailed patients saved to {OUTPUT_DIR}/failed_patients.csv')

In [ ]:
# ── Visual spot-check ────────────────────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image

# Show a few tumor slices from the first patient with tumor data
tumor_df = df[df['has_tumor']]
if len(tumor_df) > 0:
    sample_patient = tumor_df['patient_id'].iloc[0]
    sample = tumor_df[tumor_df['patient_id'] == sample_patient].head(4)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, (_, row) in enumerate(sample.iterrows()):
        if i >= 4:
            break
        img = Image.open(os.path.join(OUTPUT_DIR, row['path_image']))
        axes[0, i].imshow(img, cmap='gray')
        axes[0, i].set_title(f"{row['patient_id']}\nslice {row['slice_index']}")
        axes[0, i].axis('off')

        if row['path_mask']:
            mask = Image.open(os.path.join(OUTPUT_DIR, row['path_mask']))
            axes[1, i].imshow(mask, cmap='gray')
            axes[1, i].set_title('Tumor mask')
        axes[1, i].axis('off')

    plt.suptitle(f'Spot Check — {sample_patient}', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No tumor slices found for visual check')